# 📊 Notebook 05: Evaluación del Modelo Bayesiano
**Proyecto:** Análisis de Sentimiento Bayesiano con Calibración de Confianza

Este notebook evalúa el pipeline completo (LDA + BNN) sobre el conjunto de test. Se calculan las siguientes métricas **offline**:

1. **Accuracy** — línea base de rendimiento discriminativo
2. **NLL (Negative Log-Likelihood)** — calidad de la distribución predictiva p(y|x)
3. **ECE (Expected Calibration Error)** — calibración de la confianza del modelo
4. **Descomposición de incertidumbre** — aleatórica vs. epistémica
5. **Curva de Rechazo** — precisión/cobertura al aplicar el umbral τ sobre la entropía
6. **Detección OOD** — AUROC usando incertidumbre epistémica como señal de anomalía

## 0. Setup e imports

In [1]:
import os
import sys
import json
import numpy as np
import torch
import pyro
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve
from pyro.infer import Predictive

# Ajuste de ruta
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
root_path = os.getcwd()
if root_path not in sys.path:
    sys.path.append(root_path)

from src.data.loader import make_dataloaders, load_tfidf_splits, load_ood_embeddings
from src.models.lda import TopicModeler
from src.models.bnn import BayesianClassifier
from src.inference.variational import VariationalInference

sns.set_theme(style='whitegrid')
%matplotlib inline

pyro.clear_param_store()
print('✅ Imports completados.')

/Users/emmarey/Library/Mobile Documents/com~apple~CloudDocs/Documents/ICAI/MÁSTER/SEGUNDO CUATRI/IA PROBABILISTCA/Bayesian-Sentiment-Analysis/ia_prob_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports completados.


## 1. Carga del modelo entrenado y construcción del test loader

**Nota crítica:** `make_dataloaders` necesita que `topic_vecs` tenga tamaño N total (todos los splits), no solo el test set. Por eso regeneramos theta para todos los splits y los apilamos antes de llamar al loader.

In [2]:
# --- 1a. Cargar LDA y generar theta para TODOS los splits (crítico para el loader) ---
lda_modeler = TopicModeler.load('experiments/results/lda_model.pkl')

X_tr, X_val, X_te, _ = load_tfidf_splits()
theta_tr  = lda_modeler.get_topics(X_tr)
theta_val = lda_modeler.get_topics(X_val)
theta_te  = lda_modeler.get_topics(X_te)

# Apilamos en el mismo orden que el loader espera (train, val, test)
all_theta = np.vstack([theta_tr, theta_val, theta_te])

# --- 1b. Construir loader pasando all_theta (N_total x K) ---
_, _, test_loader = make_dataloaders(batch_size=64, topic_vecs=all_theta)

# --- 1c. Instanciar modelo y cargar pesos de la posterior ---
input_dim = 768 + lda_modeler.n_components  # 768 + K
model = BayesianClassifier(input_dim=input_dim)
vi_engine = VariationalInference(model)

state = torch.load('experiments/results/bnn_params.pt', weights_only=False)
pyro.get_param_store().set_state(state)

print(f'✅ Modelo cargado. input_dim={input_dim}')
print(f'   Test batches: {len(test_loader)}')

✅ Modelo cargado. input_dim=778
   Test batches: 16


## 2. Inferencia estocástica (Monte Carlo sobre la posterior)

Usamos `pyro.infer.Predictive` para muestrear correctamente de la posterior q(w|φ) aprendida durante el entrenamiento SVI. Cada llamada muestrea una red neuronal completa del espacio de pesos.

Con T=50 muestras obtenemos:
- **p̄(y|x)** = media de las distribuciones predictivas → predicción final
- **Incertidumbre total** H[p̄(y|x)] = entropía de la predictiva marginalizada
- **Incertidumbre aleatórica** = E_q[H[p(y|x,w)]] = ruido irreducible del lenguaje
- **Incertidumbre epistémica** = total − aleatórica = desconocimiento del modelo

In [3]:
NUM_SAMPLES = 50  # muestras Monte Carlo de la posterior

predictive = Predictive(
    vi_engine.model,
    guide=vi_engine.guide,
    num_samples=NUM_SAMPLES,
    return_sites=['obs'],
)

all_mean_probs   = []  # (N, 2) — distribución predictiva media
all_total_unc    = []  # (N,)   — entropía total H[p̄(y|x)]
all_aleatoric    = []  # (N,)   — incertidumbre aleatórica
all_epistemic    = []  # (N,)   — incertidumbre epistémica
all_labels       = []  # (N,)   — etiquetas verdaderas

def entropy(probs, eps=1e-8):
    """Entropía de Shannon en bits (base 2). probs: (batch, classes)"""
    return -(probs * torch.log2(probs + eps)).sum(dim=-1)

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        # samples['obs']: (NUM_SAMPLES, batch_size)  — etiquetas predichas {0,1}
        samples = predictive(x_batch)
        obs = samples['obs'].float()  # (T, B)

        # --- Distribución predictiva completa ---
        # Para cada muestra t, construimos prob de clase 1 y clase 0
        # obs[t, b] ∈ {0, 1}  →  prob_pos[t,b] = obs[t,b]
        prob_pos = obs  # (T, B)
        prob_neg = 1.0 - obs
        # Apilar: (T, B, 2)
        sample_probs = torch.stack([prob_neg, prob_pos], dim=-1).clamp(1e-6, 1 - 1e-6)

        # p̄(y|x): media sobre las T muestras → (B, 2)
        mean_probs = sample_probs.mean(dim=0)

        # Incertidumbre total: H[p̄(y|x)]
        total_unc = entropy(mean_probs)  # (B,)

        # Incertidumbre aleatórica: E_q[H[p(y|x,w)]]
        per_sample_entropy = entropy(sample_probs)  # (T, B)
        aleatoric = per_sample_entropy.mean(dim=0)  # (B,)

        # Incertidumbre epistémica: MI = total − aleatórica
        epistemic = (total_unc - aleatoric).clamp(min=0)  # (B,)

        all_mean_probs.append(mean_probs)
        all_total_unc.append(total_unc)
        all_aleatoric.append(aleatoric)
        all_epistemic.append(epistemic)
        all_labels.append(y_batch)

mean_probs = torch.cat(all_mean_probs)   # (N, 2)
total_unc  = torch.cat(all_total_unc)    # (N,)
aleatoric  = torch.cat(all_aleatoric)    # (N,)
epistemic  = torch.cat(all_epistemic)    # (N,)
true_labels = torch.cat(all_labels)      # (N,)

preds = mean_probs.argmax(dim=-1)        # (N,) — clase predicha

print(f'✅ Inferencia completada sobre {len(true_labels)} muestras.')
print(f'   Incertidumbre total media:     {total_unc.mean():.4f} bits')
print(f'   Incertidumbre aleatórica media: {aleatoric.mean():.4f} bits')
print(f'   Incertidumbre epistémica media: {epistemic.mean():.4f} bits')

ValueError: Continuous inference cannot handle discrete sample site 'obs'. Consider enumerating that variable as documented in https://pyro.ai/examples/enumeration.html . If you are already enumerating, take care to hide this site when constructing an autoguide, e.g. guide = AutoNormal(poutine.block(model, hide=['obs'])).
Trace Shapes:
 Param Sites:
Sample Sites:

## 3. Métricas offline

### 3.1 Accuracy y NLL

In [4]:
# --- Accuracy ---
accuracy = (preds == true_labels).float().mean().item()

# --- NLL (Negative Log-Likelihood) ---
# NLL = -E[log p(y_true | x)]  —  mide la calidad de la distribución predictiva completa
# Un modelo que asigna alta probabilidad a la clase correcta tiene NLL bajo
true_probs = mean_probs[torch.arange(len(true_labels)), true_labels]  # prob asignada a y_true
nll = -torch.log(true_probs + 1e-8).mean().item()

print('=' * 45)
print(f'  Accuracy :  {accuracy:.4f}  ({100*accuracy:.2f}%)')
print(f'  NLL      :  {nll:.4f}')
print('=' * 45)
print()
print('Interpretación NLL:')
print(f'  Modelo perfecto (p=1.0 siempre) → NLL = 0.0')
print(f'  Modelo aleatorio (p=0.5)         → NLL ≈ {-np.log(0.5):.4f}')
print(f'  Nuestro modelo                   → NLL = {nll:.4f}')

NameError: name 'preds' is not defined

### 3.2 ECE (Expected Calibration Error)

La ECE mide si la confianza del modelo coincide con su precisión real. Si el modelo dice "estoy 80% seguro" en un grupo de predicciones, debería acertar el 80% de las veces.

$$\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{N} \left| \text{acc}(B_m) - \text{conf}(B_m) \right|$$

donde $B_m$ son los bins de confianza. ECE cercano a 0 indica un modelo bien calibrado.

In [ ]:
def compute_ece(probs, labels, n_bins=10):
    """
    Expected Calibration Error con diagrama de fiabilidad.
    
    Args:
        probs  : tensor (N,) — probabilidad de la clase predicha (confianza)
        labels : tensor (N,) — etiquetas verdaderas
        n_bins : int — número de bins de confianza
    """
    confidences = probs.max(dim=-1).values  # confianza = max prob
    predictions = probs.argmax(dim=-1)
    correct = (predictions == labels).float()

    bin_edges = torch.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs, bin_counts = [], [], []

    for i in range(n_bins):
        lo, hi = bin_edges[i].item(), bin_edges[i + 1].item()
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            bin_accs.append(0.0)
            bin_confs.append((lo + hi) / 2)
            bin_counts.append(0)
            continue
        acc  = correct[mask].mean().item()
        conf = confidences[mask].mean().item()
        n    = mask.sum().item()
        ece += (n / len(labels)) * abs(acc - conf)
        bin_accs.append(acc)
        bin_confs.append(conf)
        bin_counts.append(n)

    return ece, bin_accs, bin_confs, bin_counts, bin_edges.tolist()


ece, bin_accs, bin_confs, bin_counts, bin_edges = compute_ece(mean_probs, true_labels)

print(f'ECE = {ece:.4f}')
print(f'(0.0 = perfectamente calibrado | 0.5 = totalmente descalibrado)')

# --- Diagrama de Fiabilidad (Reliability Diagram) ---
fig, ax = plt.subplots(figsize=(7, 6))

bin_centers = [(bin_edges[i] + bin_edges[i+1]) / 2 for i in range(len(bin_edges)-1)]
bar_width   = bin_edges[1] - bin_edges[0]

ax.bar(bin_centers, bin_accs, width=bar_width * 0.85,
       alpha=0.7, color='steelblue', label='Accuracy del modelo')
ax.plot([0, 1], [0, 1], 'r--', lw=2, label='Calibración perfecta')
ax.set_xlabel('Confianza media del bin', fontsize=12)
ax.set_ylabel('Accuracy media del bin', fontsize=12)
ax.set_title(f'Diagrama de Fiabilidad (Reliability Diagram)\nECE = {ece:.4f}', fontsize=13)
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
os.makedirs('experiments/results', exist_ok=True)
plt.savefig('experiments/results/reliability_diagram.png', dpi=150)
plt.show()

## 4. Descomposición de Incertidumbre

Visualizamos la distribución de incertidumbre aleatórica vs. epistémica sobre el test set para verificar que el modelo discrimina correctamente entre ambas fuentes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ['Incertidumbre Total\nH[p̄(y|x)]',
          'Incertidumbre Aleatórica\nE_q[H[p(y|x,w)]]',
          'Incertidumbre Epistémica\nMI = Total − Aleatórica']
data   = [total_unc.numpy(), aleatoric.numpy(), epistemic.numpy()]
colors = ['#4C72B0', '#DD8452', '#55A868']

for ax, title, d, c in zip(axes, titles, data, colors):
    ax.hist(d, bins=40, color=c, alpha=0.8, edgecolor='white')
    ax.axvline(d.mean(), color='red', linestyle='--', lw=1.5,
               label=f'Media = {d.mean():.3f}')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Bits', fontsize=10)
    ax.set_ylabel('Frecuencia', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('Descomposición de Incertidumbre en Test Set', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('experiments/results/uncertainty_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Aleatórica media  : {aleatoric.mean():.4f} bits  (ruido irreducible del lenguaje)')
print(f'Epistémica media  : {epistemic.mean():.4f} bits  (desconocimiento del modelo)')
print(f'Ratio epistémica  : {100*epistemic.mean()/total_unc.mean():.1f}% de la incertidumbre total')

## 5. Curva de Rechazo (Precisión / Cobertura)

El sistema se abstiene cuando la **entropía predictiva** H[p̄(y|x)] supera un umbral τ. A mayor τ, más muestras se aceptan (mayor cobertura) pero la precisión baja. Trazamos la curva para encontrar el punto óptimo de operación.

In [ ]:
# Barremos umbrales τ sobre la entropía total
thresholds = np.linspace(0, total_unc.max().item(), 100)
precisions, coverages = [], []

for tau in thresholds:
    accepted_mask = total_unc.numpy() <= tau
    coverage  = accepted_mask.mean()  # fracción de muestras aceptadas
    if accepted_mask.sum() == 0:
        precisions.append(1.0)
        coverages.append(0.0)
        continue
    precision = (preds[accepted_mask] == true_labels[accepted_mask]).float().mean().item()
    precisions.append(precision)
    coverages.append(coverage)

# Umbral óptimo: máxima precisión manteniendo cobertura ≥ 0.7
optimal_idx = next((i for i, (p, c) in enumerate(zip(precisions, coverages))
                    if c >= 0.70), -1)
tau_opt = thresholds[optimal_idx]
prec_opt = precisions[optimal_idx]
cov_opt  = coverages[optimal_idx]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(coverages, precisions, color='steelblue', lw=2.5, label='BNN (nuestro modelo)')
ax.axhline(accuracy, color='gray', linestyle=':', lw=1.5,
           label=f'Accuracy sin rechazo ({accuracy:.3f})')
ax.scatter([cov_opt], [prec_opt], color='red', zorder=5, s=80,
           label=f'Umbral óptimo τ={tau_opt:.3f}\n(cob={cov_opt:.2f}, prec={prec_opt:.3f})')
ax.set_xlabel('Cobertura (fracción de muestras aceptadas)', fontsize=12)
ax.set_ylabel('Precisión en muestras aceptadas', fontsize=12)
ax.set_title('Curva de Rechazo: Precisión vs. Cobertura', fontsize=13)
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('experiments/results/rejection_curve.png', dpi=150)
plt.show()

print(f'\n✅ Umbral óptimo τ = {tau_opt:.4f}')
print(f'   Cobertura   = {cov_opt:.2%}  (se acepta el {cov_opt:.0%} de las muestras)')
print(f'   Precisión   = {prec_opt:.4f}  (vs. {accuracy:.4f} sin rechazo)')
print(f'   Mejora      = +{prec_opt - accuracy:.4f} puntos de precisión al descartar muestras dudosas')

## 6. Detección OOD con Incertidumbre Epistémica (AUROC)

El modelo debería asignar **mayor incertidumbre epistémica** a las muestras OOD (fuera del dominio de entrenamiento) que a las muestras in-distribution. Medimos esto con el **AUROC** de la incertidumbre epistémica como detector binario (ID vs. OOD).

In [ ]:
# --- Cargar embeddings OOD y generar theta OOD ---
ood_bert = load_ood_embeddings()  # (N_ood, 768)

# Necesitamos theta para los embeddings OOD
# Cargamos la matriz TF-IDF OOD desde el vectorizador
from src.data.loader import load_vectorizer
import scipy.sparse as sp

with open('data/ood/ood_texts.json') as f:
    ood_texts = json.load(f)

vectorizer = load_vectorizer()
tfidf_ood  = vectorizer.transform(ood_texts)  # sparse (N_ood, V)
theta_ood  = lda_modeler.get_topics(tfidf_ood)  # (N_ood, K)

# Concatenar BERT + LDA para muestras OOD
x_ood = torch.tensor(
    np.hstack([ood_bert, theta_ood]),
    dtype=torch.float32
)

# --- Inferencia estocástica sobre muestras OOD ---
with torch.no_grad():
    samples_ood = predictive(x_ood)
    obs_ood = samples_ood['obs'].float()  # (T, N_ood)
    prob_pos_ood = obs_ood
    prob_neg_ood = 1.0 - obs_ood
    sp_ood = torch.stack([prob_neg_ood, prob_pos_ood], dim=-1).clamp(1e-6, 1-1e-6)
    mean_ood = sp_ood.mean(dim=0)
    total_ood = entropy(mean_ood)
    aleat_ood = entropy(sp_ood).mean(dim=0)
    epist_ood = (total_ood - aleat_ood).clamp(min=0)

print(f'✅ Inferencia OOD completada: {len(ood_texts)} muestras OOD')
print(f'   Incertidumbre epistémica OOD media  : {epist_ood.mean():.4f} bits')
print(f'   Incertidumbre epistémica ID media   : {epistemic.mean():.4f} bits')

In [ ]:
# --- AUROC: incertidumbre epistémica como detector OOD ---
# Etiqueta 1 = OOD, 0 = in-distribution
scores    = np.concatenate([epistemic.numpy(), epist_ood.numpy()])
ood_gt    = np.concatenate([np.zeros(len(epistemic)), np.ones(len(epist_ood))])

auroc = roc_auc_score(ood_gt, scores)
fpr, tpr, _ = roc_curve(ood_gt, scores)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC
axes[0].plot(fpr, tpr, color='steelblue', lw=2.5, label=f'AUROC = {auroc:.4f}')
axes[0].plot([0, 1], [0, 1], 'r--', lw=1.5, label='Aleatorio (AUROC=0.5)')
axes[0].set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
axes[0].set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
axes[0].set_title('Curva ROC — Detección OOD\nvía Incertidumbre Epistémica', fontsize=12)
axes[0].legend(fontsize=10)

# Distribución de incertidumbre epistémica ID vs OOD
axes[1].hist(epistemic.numpy(), bins=40, alpha=0.7,
             color='steelblue', label='In-Distribution (ID)', density=True)
axes[1].hist(epist_ood.numpy(), bins=40, alpha=0.7,
             color='salmon', label='Out-of-Distribution (OOD)', density=True)
axes[1].set_xlabel('Incertidumbre Epistémica (bits)', fontsize=12)
axes[1].set_ylabel('Densidad', fontsize=12)
axes[1].set_title('Distribución de Incertidumbre\nID vs. OOD', fontsize=12)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('experiments/results/ood_detection.png', dpi=150)
plt.show()

print(f'\n✅ AUROC detección OOD = {auroc:.4f}')
print(f'   (1.0 = detector perfecto | 0.5 = aleatorio)')

## 7. Análisis cualitativo — casos concretos

Inspeccionamos ejemplos reales del test set ordenados por incertidumbre para ilustrar el comportamiento del modelo en el paper.

In [ ]:
# Cargamos los textos originales del test set para inspeccionarlos
labels_all = np.load('data/processed/labels.npy')
with open('data/processed/splits.json') as f:
    splits = json.load(f)

# Nota: no tenemos los textos en memoria aquí, pero sí los índices
# Mostramos estadísticas por cuartil de incertidumbre

unc_np    = total_unc.numpy()
labels_np = true_labels.numpy()
preds_np  = preds.numpy()
conf_np   = mean_probs.max(dim=-1).values.numpy()

quartiles = np.percentile(unc_np, [0, 25, 50, 75, 100])

print('Análisis por cuartil de incertidumbre total\n')
print(f'{"Cuartil":<12} {"Rango H":<20} {"N muestras":<12} {"Accuracy":<12} {"Conf. media"}')
print('-' * 65)

labels_q = ['Q1 (bajo)', 'Q2', 'Q3', 'Q4 (alto)']
for i in range(4):
    lo, hi = quartiles[i], quartiles[i+1]
    mask = (unc_np >= lo) & (unc_np <= hi)
    if mask.sum() == 0:
        continue
    acc_q  = (preds_np[mask] == labels_np[mask]).mean()
    conf_q = conf_np[mask].mean()
    print(f'{labels_q[i]:<12} [{lo:.3f}, {hi:.3f}]    {mask.sum():<12} {acc_q:.4f}       {conf_q:.4f}')

## 8. Resumen de métricas

In [ ]:
print('=' * 55)
print('  RESUMEN DE MÉTRICAS — BAYESIAN SENTIMENT ANALYSIS')
print('=' * 55)
print(f'  Accuracy                    : {accuracy:.4f}')
print(f'  NLL (↓ mejor)               : {nll:.4f}')
print(f'  ECE (↓ mejor, 0=perfecto)   : {ece:.4f}')
print(f'  AUROC detección OOD (↑)     : {auroc:.4f}')
print(f'  Umbral rechazo óptimo τ     : {tau_opt:.4f}')
print(f'  Precisión con rechazo       : {prec_opt:.4f}  (+{prec_opt-accuracy:.4f})')
print(f'  Cobertura con τ óptimo      : {cov_opt:.2%}')
print('=' * 55)
print()
print('Figuras guardadas en experiments/results/:')
print('  - reliability_diagram.png')
print('  - uncertainty_decomposition.png')
print('  - rejection_curve.png')
print('  - ood_detection.png')